In [300]:
%pip install -q nfl-data-py

import pandas as pd
import numpy as np
import nfl_data_py as nfl
import nfl_td_lambda.data_collection as data
nfl_teams = pd.read_csv('nfl_teams.csv')
team_map = dict(zip(nfl_teams['team_name'], nfl_teams['team_id']))


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [301]:
from datetime import datetime

# Set the season to analyze (e.g., 2024). Change if needed.
season = 2025
week = 1

# Load play-by-play data for the chosen season
def get_weekly_scorers(season, week):
    pbp = nfl.import_pbp_data(years=[season])

    # Filter for regular season, Week 1
    week_pbp = pbp[(pbp['week'] == week)]

    # Keep only touchdown plays
    week_tds = week_pbp[week_pbp['touchdown'] == 1]

    # Count TDs per scorer. Prefer id+name if both available, else fall back to name only
    use_cols = [c for c in ['td_player_id', 'td_player_name'] if c in week_tds.columns]
    if use_cols:
        scorers = (
            week_tds.dropna(subset=use_cols)
            .groupby(use_cols)
            .size()
            .reset_index(name='tds')
        )
        if 'td_player_id' in use_cols:
            scorers = scorers.rename(columns={'td_player_name': 'player', 'td_player_id': 'player_id'})
        else:
            scorers = scorers.rename(columns={'td_player_name': 'player'})
    else:
        # Fallback if td_* columns not present; derive from rusher/receiver
        rush = week_tds.dropna(subset=['rusher_player_id'])[['rusher_player_id', 'rusher_player_name']]
        rec = week_tds.dropna(subset=['receiver_player_id'])[['receiver_player_id', 'receiver_player_name']]
        rush.columns = ['player_id', 'player']
        rec.columns = ['player_id', 'player']
        both = pd.concat([rush, rec], ignore_index=True)
        scorers = both.groupby(['player_id', 'player']).size().reset_index(name='tds')

    return scorers

    # Show results
scorers = get_weekly_scorers(2025, week)
scorers.head(50)


2025 done.
Downcasting floats.


,player_id,player,tds
0,00-0030061,Z.Ertz,1
1,00-0030279,K.Allen,1
2,00-0030506,T.Kelce,1
3,00-0030564,D.Hopkins,1
4,00-0032764,D.Henry,2
5,00-0033288,G.Kittle,1
6,00-0033293,A.Jones,1
7,00-0033553,J.Conner,1
8,00-0033858,J.Smith,1
9,00-0033873,P.Mahomes,1


In [302]:
predictions = pd.read_csv(f'data/predictions_week_{week}.csv')

#Keep only player_id, player_display_name, predicted_touchdown_probability, and model_edge
predictions = predictions[['player_id', 'player_display_name', 'position','team', 'predicted_touchdown_probability', 'price', 'model_edge','market_implied_prob']]

#Sort by predicted_touchdown_probability in descending order
predictions = predictions.sort_values(by='predicted_touchdown_probability', ascending=False)

#Show the top 50 players
predictions.head(50)



,player_id,player_display_name,position,team,predicted_touchdown_probability,price,model_edge,market_implied_prob
0,00-0032764,Derrick Henry,RB,BAL,0.567211,-145.0,-0.024626,0.591837
1,00-0036158,J.K. Dobbins,RB,DEN,0.550673,160.0,0.166058,0.384615
2,00-0034844,Saquon Barkley,RB,PHI,0.550125,-185.0,-0.098998,0.649123
3,00-0036223,Jonathan Taylor,RB,IND,0.548110,-180.0,-0.094748,0.642857
4,00-0035700,Josh Jacobs,RB,GB,0.538403,-160.0,-0.076982,0.615385
5,00-0033553,James Conner,RB,ARI,0.534307,-155.0,-0.073536,0.607843
6,00-0039361,Bucky Irving,RB,TB,0.534180,-140.0,-0.049154,0.583333
7,00-0039139,Jahmyr Gibbs,RB,DET,0.528690,-105.0,0.016495,0.512195
8,00-0039040,De'Von Achane,RB,MIA,0.520523,-140.0,-0.062810,0.583333
9,00-0033293,Aaron Jones,RB,MIN,0.499995,165.0,0.122637,0.377358


In [303]:
# Join predictions with scorers: prefer player_id, else fall back to name
if 'player_id' in scorers.columns:
    pred_scored = predictions.merge(
        scorers[['player_id', 'tds']], on='player_id', how='inner'
    )
else:
    pred_scored = predictions.merge(
        scorers[['player', 'tds']], left_on='player_display_name', right_on='player', how='inner'
    )

pred_scored.sort_values(['predicted_touchdown_probability'], ascending=[False])

,player_id,player_display_name,position,team,predicted_touchdown_probability,price,model_edge,market_implied_prob,tds
0,00-0032764,Derrick Henry,RB,BAL,0.567211,-145.0,-0.024626,0.591837,2
1,00-0036158,J.K. Dobbins,RB,DEN,0.550673,160.0,0.166058,0.384615,1
2,00-0034844,Saquon Barkley,RB,PHI,0.550125,-185.0,-0.098998,0.649123,1
3,00-0035700,Josh Jacobs,RB,GB,0.538403,-160.0,-0.076982,0.615385,1
4,00-0033553,James Conner,RB,ARI,0.534307,-155.0,-0.073536,0.607843,1
5,00-0039361,Bucky Irving,RB,TB,0.534180,-140.0,-0.049154,0.583333,1
6,00-0039040,De'Von Achane,RB,MIA,0.520523,-140.0,-0.062810,0.583333,1
7,00-0033293,Aaron Jones,RB,MIN,0.499995,165.0,0.122637,0.377358,1
8,00-0038597,Chase Brown,RB,CIN,0.486931,-150.0,-0.113069,0.600000,1
9,00-0037840,Kyren Williams,RB,LA,0.484429,-140.0,-0.098904,0.583333,1


In [304]:
###Simulate betting on the top 10 running backs
def simulate_betting(df, scorers):
    stake = 10.0

    bets = df.copy()

    # Merge to mark hits
    bets = bets.merge(
        scorers[['player_id', 'tds']], on='player_id', how='left'
    )
    bets['tds'] = bets['tds'].fillna(0).astype(int)
    bets['hit'] = bets['tds'] > 0

    # American odds payout logic
    # profit_if_win = stake * (odds/100) if odds > 0 else stake * (100/abs(odds))
    # profit_if_loss = -stake
    is_plus = bets['price'] > 0
    profit_if_win = stake * (bets['price'] / 100.0)
    profit_if_win = profit_if_win.where(is_plus, stake * (100.0 / bets['price'].abs()))

    bets['profit'] = np.where(bets['hit'], profit_if_win, -stake)

    # Add total return
    bets['return'] = stake + bets['profit']

    # Summary metrics
    num_bets = len(bets)
    hits = int(bets['hit'].sum())
    hit_rate = hits / num_bets if num_bets else 0.0
    total_profit = float(bets['profit'].sum())
    roi = total_profit / (stake * num_bets) if num_bets else 0.0

    summary = {
        'bets': num_bets,
        'hits': hits,
        'hit_rate': round(hit_rate, 3),
        'total_profit': round(total_profit, 2),
        'roi': round(roi, 3)
    }

    display(summary)

    # Show detailed results
    cols = [
        'player_id', 'player_display_name', 'team', 'position', 'price',
        'predicted_touchdown_probability', 'model_edge', 'tds', 'hit', 'profit', 'return'
    ]
    return bets[cols].sort_values(['predicted_touchdown_probability'], ascending=[False]).reset_index(drop=True)



In [415]:
# output players from predictions that play for 'PHI', 'KC', 'LAC' or 'DAL'

#find players with model_edge > 0 and price < 500
ev = predictions[predictions['model_edge'] >= 0.05] 
ev = ev[ev['price'] <= 400]

#sort by model_edge in descending order
ev = ev.sort_values('model_edge', ascending=False)
ev







,player_id,player_display_name,position,team,predicted_touchdown_probability,price,model_edge,market_implied_prob
18,00-0036139,Rico Dowdle,RB,CAR,0.420521,320.0,0.182426,0.238095
36,00-0038544,Quentin Johnston,WR,LAC,0.385825,370.0,0.173059,0.212766
1,00-0036158,J.K. Dobbins,RB,DEN,0.550673,160.0,0.166058,0.384615
41,00-0040242,Jacory Croskey-Merritt,RB,WAS,0.374350,330.0,0.141792,0.232558
55,00-0033923,Kareem Hunt,RB,KC,0.346429,370.0,0.133663,0.212766
39,00-0030035,Adam Thielen,WR,MIN,0.377531,310.0,0.133628,0.243902
46,00-0037256,Rachaad White,RB,TB,0.362283,320.0,0.124188,0.238095
9,00-0033293,Aaron Jones,RB,MIN,0.499995,165.0,0.122637,0.377358
76,00-0038117,Wan'Dale Robinson,WR,NYG,0.296851,400.0,0.096851,0.200000
16,00-0036912,DeVonta Smith,WR,PHI,0.432631,180.0,0.075488,0.357143


In [416]:
simulate_betting(ev, scorers)

{'bets': 19, 'hits': 8, 'hit_rate': 0.421, 'total_profit': 118.5, 'roi': 0.624}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0036158,J.K. Dobbins,DEN,RB,160.0,0.550673,0.166058,1,True,16.0,26.0
1,00-0033293,Aaron Jones,MIN,RB,165.0,0.499995,0.122637,1,True,16.5,26.5
2,00-0038120,Breece Hall,NYJ,RB,170.0,0.444193,0.073823,0,False,-10.0,0.0
3,00-0036912,DeVonta Smith,PHI,WR,180.0,0.432631,0.075488,0,False,-10.0,0.0
4,00-0036139,Rico Dowdle,CAR,RB,320.0,0.420521,0.182426,0,False,-10.0,0.0
5,00-0037744,Trey McBride,ARI,TE,200.0,0.402582,0.069248,0,False,-10.0,0.0
6,00-0036252,Michael Pittman,IND,WR,215.0,0.390195,0.072734,1,True,21.5,31.5
7,00-0038544,Quentin Johnston,LAC,WR,370.0,0.385825,0.173059,2,True,37.0,47.0
8,00-0034960,Jakobi Meyers,LV,WR,225.0,0.380881,0.073189,0,False,-10.0,0.0
9,00-0030035,Adam Thielen,MIN,WR,310.0,0.377531,0.133628,0,False,-10.0,0.0


In [417]:
### Get top 10 rb, wr, te, qb from predictions
top_rb = predictions[predictions['position'] == 'RB'].sort_values('predicted_touchdown_probability', ascending=False).head(10)
top_wr = predictions[predictions['position'] == 'WR'].sort_values('predicted_touchdown_probability', ascending=False).head(10)
top_te = predictions[predictions['position'] == 'TE'].sort_values('predicted_touchdown_probability', ascending=False).head(5)
top_qb = predictions[predictions['position'] == 'QB'].sort_values('predicted_touchdown_probability', ascending=False).head(5)

top_rb



,player_id,player_display_name,position,team,predicted_touchdown_probability,price,model_edge,market_implied_prob
0,00-0032764,Derrick Henry,RB,BAL,0.567211,-145.0,-0.024626,0.591837
1,00-0036158,J.K. Dobbins,RB,DEN,0.550673,160.0,0.166058,0.384615
2,00-0034844,Saquon Barkley,RB,PHI,0.550125,-185.0,-0.098998,0.649123
3,00-0036223,Jonathan Taylor,RB,IND,0.548110,-180.0,-0.094748,0.642857
4,00-0035700,Josh Jacobs,RB,GB,0.538403,-160.0,-0.076982,0.615385
5,00-0033553,James Conner,RB,ARI,0.534307,-155.0,-0.073536,0.607843
6,00-0039361,Bucky Irving,RB,TB,0.534180,-140.0,-0.049154,0.583333
7,00-0039139,Jahmyr Gibbs,RB,DET,0.528690,-105.0,0.016495,0.512195
8,00-0039040,De'Von Achane,RB,MIA,0.520523,-140.0,-0.062810,0.583333
9,00-0033293,Aaron Jones,RB,MIN,0.499995,165.0,0.122637,0.377358


In [418]:
simulate_betting(top_rb, scorers)


{'bets': 10, 'hits': 8, 'hit_rate': 0.8, 'total_profit': 51.79, 'roi': 0.518}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0032764,Derrick Henry,BAL,RB,-145.0,0.567211,-0.024626,2,True,6.896552,16.896552
1,00-0036158,J.K. Dobbins,DEN,RB,160.0,0.550673,0.166058,1,True,16.000000,26.000000
2,00-0034844,Saquon Barkley,PHI,RB,-185.0,0.550125,-0.098998,1,True,5.405405,15.405405
3,00-0036223,Jonathan Taylor,IND,RB,-180.0,0.548110,-0.094748,0,False,-10.000000,0.000000
4,00-0035700,Josh Jacobs,GB,RB,-160.0,0.538403,-0.076982,1,True,6.250000,16.250000
5,00-0033553,James Conner,ARI,RB,-155.0,0.534307,-0.073536,1,True,6.451613,16.451613
6,00-0039361,Bucky Irving,TB,RB,-140.0,0.534180,-0.049154,1,True,7.142857,17.142857
7,00-0039139,Jahmyr Gibbs,DET,RB,-105.0,0.528690,0.016495,0,False,-10.000000,0.000000
8,00-0039040,De'Von Achane,MIA,RB,-140.0,0.520523,-0.062810,1,True,7.142857,17.142857
9,00-0033293,Aaron Jones,MIN,RB,165.0,0.499995,0.122637,1,True,16.500000,26.500000


In [419]:
#filter top_wr to only include players with model_edge > 0.05 and price < 500
#top_wr = top_wr[top_wr['model_edge'] > 0.05]
#top_wr = top_wr[top_wr['price'] < 500]
simulate_betting(top_wr, scorers)


{'bets': 10, 'hits': 3, 'hit_rate': 0.3, 'total_profit': -30.0, 'roi': -0.3}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0036900,Ja'Marr Chase,CIN,WR,-130.0,0.449069,-0.116149,0,False,-10.0,0.0
1,00-0036912,DeVonta Smith,PHI,WR,180.0,0.432631,0.075488,0,False,-10.0,0.0
2,00-0031408,Mike Evans,TB,WR,110.0,0.420342,-0.055849,0,False,-10.0,0.0
3,00-0036410,Tee Higgins,CIN,WR,120.0,0.415667,-0.038878,0,False,-10.0,0.0
4,00-0035676,A.J. Brown,PHI,WR,160.0,0.410783,0.026168,0,False,-10.0,0.0
5,00-0036963,Amon-Ra St. Brown,DET,WR,140.0,0.407595,-0.009072,0,False,-10.0,0.0
6,00-0039915,Ladd McConkey,LAC,WR,180.0,0.404323,0.047180,0,False,-10.0,0.0
7,00-0036322,Justin Jefferson,MIN,WR,135.0,0.403716,-0.021816,1,True,13.5,23.5
8,00-0034348,Courtland Sutton,DEN,WR,135.0,0.402520,-0.023011,1,True,13.5,23.5
9,00-0039893,Brian Thomas Jr.,JAX,WR,130.0,0.399390,-0.035393,1,True,13.0,23.0


In [420]:
simulate_betting(top_te, scorers)

{'bets': 5, 'hits': 1, 'hit_rate': 0.2, 'total_profit': -23.5, 'roi': -0.47}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0037744,Trey McBride,ARI,TE,200.0,0.402582,0.069248,0,False,-10.0,0.0
1,00-0039338,Brock Bowers,LV,TE,190.0,0.365556,0.020729,0,False,-10.0,0.0
2,00-0030506,Travis Kelce,KC,TE,165.0,0.351173,-0.026186,1,True,16.5,26.5
3,00-0039065,Sam LaPorta,DET,TE,235.0,0.347692,0.049185,0,False,-10.0,0.0
4,00-0033885,David Njoku,CLE,TE,230.0,0.327245,0.024215,0,False,-10.0,0.0


In [421]:
simulate_betting(top_qb, scorers)

{'bets': 5, 'hits': 5, 'hit_rate': 1.0, 'total_profit': 114.5, 'roi': 2.29}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0034857,Josh Allen,BUF,QB,-120.0,0.389435,-0.156020,2,True,8.333333,18.333333
1,00-0036389,Jalen Hurts,PHI,QB,-150.0,0.327158,-0.272842,2,True,6.666667,16.666667
2,00-0034796,Lamar Jackson,BAL,QB,205.0,0.310611,-0.017258,1,True,20.500000,30.500000
3,00-0035710,Daniel Jones,IND,QB,190.0,0.276491,-0.068337,2,True,19.000000,29.000000
4,00-0039923,J.J. McCarthy,MIN,QB,600.0,0.250541,0.107684,1,True,60.000000,70.000000


In [422]:
top_predictors = predictions[predictions['predicted_touchdown_probability'] >= 0.50]
simulate_betting(top_predictors, scorers)

{'bets': 9, 'hits': 7, 'hit_rate': 0.778, 'total_profit': 35.29, 'roi': 0.392}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0032764,Derrick Henry,BAL,RB,-145.0,0.567211,-0.024626,2,True,6.896552,16.896552
1,00-0036158,J.K. Dobbins,DEN,RB,160.0,0.550673,0.166058,1,True,16.000000,26.000000
2,00-0034844,Saquon Barkley,PHI,RB,-185.0,0.550125,-0.098998,1,True,5.405405,15.405405
3,00-0036223,Jonathan Taylor,IND,RB,-180.0,0.548110,-0.094748,0,False,-10.000000,0.000000
4,00-0035700,Josh Jacobs,GB,RB,-160.0,0.538403,-0.076982,1,True,6.250000,16.250000
5,00-0033553,James Conner,ARI,RB,-155.0,0.534307,-0.073536,1,True,6.451613,16.451613
6,00-0039361,Bucky Irving,TB,RB,-140.0,0.534180,-0.049154,1,True,7.142857,17.142857
7,00-0039139,Jahmyr Gibbs,DET,RB,-105.0,0.528690,0.016495,0,False,-10.000000,0.000000
8,00-0039040,De'Von Achane,MIA,RB,-140.0,0.520523,-0.062810,1,True,7.142857,17.142857


In [423]:
top_vegas = predictions.sort_values('market_implied_prob', ascending=False).head(19)
simulate_betting(top_vegas, scorers)

{'bets': 19,
 'hits': 15,
 'hit_rate': 0.789,
 'total_profit': 71.69,
 'roi': 0.377}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0032764,Derrick Henry,BAL,RB,-145.0,0.567211,-0.024626,2,True,6.896552,16.896552
1,00-0034844,Saquon Barkley,PHI,RB,-185.0,0.550125,-0.098998,1,True,5.405405,15.405405
2,00-0036223,Jonathan Taylor,IND,RB,-180.0,0.548110,-0.094748,0,False,-10.000000,0.000000
3,00-0035700,Josh Jacobs,GB,RB,-160.0,0.538403,-0.076982,1,True,6.250000,16.250000
4,00-0033553,James Conner,ARI,RB,-155.0,0.534307,-0.073536,1,True,6.451613,16.451613
5,00-0039361,Bucky Irving,TB,RB,-140.0,0.534180,-0.049154,1,True,7.142857,17.142857
6,00-0039139,Jahmyr Gibbs,DET,RB,-105.0,0.528690,0.016495,0,False,-10.000000,0.000000
7,00-0039040,De'Von Achane,MIA,RB,-140.0,0.520523,-0.062810,1,True,7.142857,17.142857
8,00-0038597,Chase Brown,CIN,RB,-150.0,0.486931,-0.113069,1,True,6.666667,16.666667
9,00-0037840,Kyren Williams,LA,RB,-140.0,0.484429,-0.098904,1,True,7.142857,17.142857


In [424]:
vegas_similar = predictions[predictions['model_edge'] < 0.05]
vegas_similar = vegas_similar[vegas_similar['model_edge'] > 0]
vegas_similar = vegas_similar[vegas_similar['price'] < 300]
simulate_betting(vegas_similar, scorers)

{'bets': 15,
 'hits': 2,
 'hit_rate': 0.133,
 'total_profit': -80.0,
 'roi': -0.533}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0039139,Jahmyr Gibbs,DET,RB,-105.0,0.528690,0.016495,0,False,-10.0,0.0
1,00-0035676,A.J. Brown,PHI,WR,160.0,0.410783,0.026168,0,False,-10.0,0.0
2,00-0039915,Ladd McConkey,LAC,WR,180.0,0.404323,0.047180,0,False,-10.0,0.0
3,00-0036875,Rhamondre Stevenson,NE,RB,155.0,0.396964,0.004807,0,False,-10.0,0.0
4,00-0036407,Jerry Jeudy,CLE,WR,200.0,0.370062,0.036728,0,False,-10.0,0.0
5,00-0034827,DJ Moore,CHI,WR,195.0,0.366720,0.027737,0,False,-10.0,0.0
6,00-0039338,Brock Bowers,LV,TE,190.0,0.365556,0.020729,0,False,-10.0,0.0
7,00-0039065,Sam LaPorta,DET,TE,235.0,0.347692,0.049185,0,False,-10.0,0.0
8,00-0037240,Jameson Williams,DET,WR,220.0,0.346483,0.033983,0,False,-10.0,0.0
9,00-0037740,Garrett Wilson,NYJ,WR,220.0,0.334791,0.022291,1,True,22.0,32.0


In [425]:
top_25 = predictions.head(25)
simulate_betting(top_25, scorers)

{'bets': 25,
 'hits': 13,
 'hit_rate': 0.52,
 'total_profit': -8.66,
 'roi': -0.035}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0032764,Derrick Henry,BAL,RB,-145.0,0.567211,-0.024626,2,True,6.896552,16.896552
1,00-0036158,J.K. Dobbins,DEN,RB,160.0,0.550673,0.166058,1,True,16.000000,26.000000
2,00-0034844,Saquon Barkley,PHI,RB,-185.0,0.550125,-0.098998,1,True,5.405405,15.405405
3,00-0036223,Jonathan Taylor,IND,RB,-180.0,0.548110,-0.094748,0,False,-10.000000,0.000000
4,00-0035700,Josh Jacobs,GB,RB,-160.0,0.538403,-0.076982,1,True,6.250000,16.250000
5,00-0033553,James Conner,ARI,RB,-155.0,0.534307,-0.073536,1,True,6.451613,16.451613
6,00-0039361,Bucky Irving,TB,RB,-140.0,0.534180,-0.049154,1,True,7.142857,17.142857
7,00-0039139,Jahmyr Gibbs,DET,RB,-105.0,0.528690,0.016495,0,False,-10.000000,0.000000
8,00-0039040,De'Von Achane,MIA,RB,-140.0,0.520523,-0.062810,1,True,7.142857,17.142857
9,00-0033293,Aaron Jones,MIN,RB,165.0,0.499995,0.122637,1,True,16.500000,26.500000


In [426]:
week_2 = pd.read_csv('data/predictions_week_2.csv')
week_2.sort_values('market_implied_prob', ascending=False).head(16)




,player_id,player_display_name,position,team,opponent_team,avg_avg_expected_yac,avg_avg_intended_air_yards,avg_avg_separation,avg_avg_time_to_los,avg_avg_yac_above_expectation,...,rush_matchup_value,pass_matchup_value,season,week,depth_chart_rank,predicted_touchdown_probability,merge_name,price,market_implied_prob,model_edge
0,00-0032764,Derrick Henry,RB,BAL,CLE,0.000000,0.000000,0.000000,2.751695,0.000000,...,0.396901,0.000062,2025.0,2.0,1,0.581738,derrick henry,-205.0,0.672131,-0.090393
3,00-0035700,Josh Jacobs,RB,GB,WAS,0.000000,0.000000,0.000000,2.210247,0.000000,...,0.466811,0.000211,2025.0,2.0,1,0.557552,josh jacobs,-200.0,0.666667,-0.109115
18,00-0033280,Christian McCaffrey,RB,SF,NO,0.000000,0.000000,0.000000,2.199097,0.000000,...,0.446241,0.108229,2025.0,2.0,1,0.422890,christian mccaffrey,-180.0,0.642857,-0.219967
4,00-0037840,Kyren Williams,RB,LA,TEN,0.000000,0.000000,0.000000,2.681487,0.000000,...,0.740778,0.004203,2025.0,2.0,1,0.556595,kyren williams,-175.0,0.636364,-0.079769
1,00-0039139,Jahmyr Gibbs,RB,DET,CHI,0.000000,0.000000,0.000000,1.722130,0.000000,...,0.599922,0.043777,2025.0,2.0,1,0.571330,jahmyr gibbs,-170.0,0.629630,-0.058300
5,00-0033553,James Conner,RB,ARI,CAR,0.000000,0.000000,0.000000,2.127841,0.000000,...,0.315327,0.020460,2025.0,2.0,1,0.544659,james conner,-155.0,0.607843,-0.063184
7,00-0039040,De'Von Achane,RB,MIA,NE,0.000000,0.000000,0.000000,1.987127,0.000000,...,0.353250,0.045626,2025.0,2.0,1,0.523978,devon achane,-155.0,0.607843,-0.083866
10,00-0038542,Bijan Robinson,RB,ATL,MIN,0.000000,0.000000,0.000000,2.896846,0.000000,...,0.700539,0.014752,2025.0,2.0,1,0.500861,bijan robinson,-155.0,0.607843,-0.106982
2,00-0034844,Saquon Barkley,RB,PHI,KC,0.000000,0.000000,0.000000,2.926910,0.000000,...,0.233517,0.000173,2025.0,2.0,1,0.562689,saquon barkley,-150.0,0.600000,-0.037311
8,00-0038597,Chase Brown,RB,CIN,JAX,0.000000,0.000000,0.000000,2.763189,0.000000,...,0.591269,0.045418,2025.0,2.0,1,0.502281,chase brown,-150.0,0.600000,-0.097719


In [427]:
#### Add week_2_lines.csv to historic_lines.csv
historic_lines = pd.read_csv('data/historic_lines.csv')
week_2_lines = pd.read_csv('data/week_2_lines.csv')







In [428]:
import pandas as pd
from typing import Optional


def append_week_lines_to_historic(
    week_lines_csv: str = "data/week_2_lines.csv",
    historic_csv: str = "data/historic_lines.csv",
    teams_csv: str = "nfl_teams.csv",
    schedule_season: int = 2025,
    schedule_week: int = 2,
    bookmaker_preference: Optional[str] = "DraftKings"
) -> int:
    """Append weekly spread lines into historic_lines.csv, preserving schema.

    - Reads week-level lines in the format of week_2_lines.csv (two rows/team per game).
    - Maps full team names to IDs matching historic_lines.csv via nfl_teams.csv.
    - Aggregates to one row per game: picks the favorite (negative point), keeps total.
    - Appends new rows to historic_lines.csv with the same columns and blank index header.

    Returns
    -------
    int
        Number of rows appended (deduped against existing season/week/home/away).
    """
    # Load team mapping (full name -> team_id used in historic file)
    teams_df = pd.read_csv(teams_csv)
    name_to_id = dict(zip(teams_df["team_name"].astype(str), teams_df["team_id"].astype(str)))

    # Load week lines and filter to spreads (and bookmaker if provided)
    week_df = pd.read_csv(week_lines_csv)
    if "market" in week_df.columns:
        week_df = week_df[week_df["market"].str.lower() == "spreads"].copy()
    if bookmaker_preference and "bookmaker" in week_df.columns:
        week_df = week_df[week_df["bookmaker"].astype(str) == bookmaker_preference].copy()

    # Normalize numeric fields
    if "point" in week_df.columns:
        week_df["point"] = pd.to_numeric(week_df["point"], errors="coerce")
    # 'over/under' has a slash in the name; keep safe access
    ou_col = "over/under" if "over/under" in week_df.columns else (
        "over_under" if "over_under" in week_df.columns else None
    )
    if ou_col is not None:
        week_df[ou_col] = pd.to_numeric(week_df[ou_col], errors="coerce")

    # Group to one row per game
    required_cols = {"game_id", "home_team", "away_team", "label"}
    missing = [c for c in required_cols if c not in week_df.columns]
    if missing:
        raise ValueError(f"Missing required columns in {week_lines_csv}: {missing}")

    records = []
    for game_id, grp in week_df.groupby("game_id", sort=False):
        home_team_name = str(grp["home_team"].iloc[0])
        away_team_name = str(grp["away_team"].iloc[0])

        # Determine favorite: row with the most negative spread (minimum point)
        grp_nonnull = grp.dropna(subset=["point"]) if "point" in grp.columns else grp.copy()
        if grp_nonnull.empty:
            # If we can't determine a favorite, skip this game
            continue
        fav_idx = grp_nonnull["point"].idxmin()
        fav_team_name = str(grp_nonnull.loc[fav_idx, "label"])  # team name in the bet label
        spread_favorite = float(grp_nonnull.loc[fav_idx, "point"])  # should be negative

        # Over/Under line: take first non-null within the game
        if ou_col is not None:
            ou_series = grp_nonnull[ou_col].dropna()
            over_under_line = float(ou_series.iloc[0]) if not ou_series.empty else None
        else:
            over_under_line = None

        # Map to team IDs used in historic file
        home_id = name_to_id.get(home_team_name)
        away_id = name_to_id.get(away_team_name)
        fav_id = name_to_id.get(fav_team_name)

        if home_id is None or away_id is None or fav_id is None:
            # Try a couple of common aliases
            alias = {
                "LA Rams": "Los Angeles Rams",
                "LA Chargers": "Los Angeles Chargers",
                "LV Raiders": "Las Vegas Raiders",
                "Washington": "Washington Commanders",
            }
            home_id = home_id or name_to_id.get(alias.get(home_team_name, home_team_name))
            away_id = away_id or name_to_id.get(alias.get(away_team_name, away_team_name))
            fav_id = fav_id or name_to_id.get(alias.get(fav_team_name, fav_team_name))

        if home_id is None or away_id is None or fav_id is None:
            raise KeyError(
                f"Missing team_id mapping. home='{home_team_name}'->{home_id}, "
                f"away='{away_team_name}'->{away_id}, favorite='{fav_team_name}'->{fav_id}"
            )

        records.append({
            "schedule_season": int(schedule_season),
            "schedule_week": int(schedule_week),
            "team_home": home_team_name,
            "team_away": away_team_name,
            "team_favorite_id": fav_id,
            "spread_favorite": spread_favorite,
            "over_under_line": over_under_line,
            "schedule_playoff": False,
            "team_home_id": home_id,
            "team_away_id": away_id,
        })

    new_rows_df = pd.DataFrame.from_records(records)
    if new_rows_df.empty:
        return 0

    # Load historic lines with existing index (blank header) and dedupe by key
    historic_df = pd.read_csv(historic_csv, index_col=0, low_memory=False)
    historic_df.index.name = ""  # Ensure blank header on index when saving

    new_rows_df["__key"] = (
        new_rows_df["schedule_season"].astype(str)
        + "|" + new_rows_df["schedule_week"].astype(str)
        + "|" + new_rows_df["team_home"].astype(str)
        + "|" + new_rows_df["team_away"].astype(str)
    )
    hist_keys = set(
        (historic_df["schedule_season"].astype(str)
         + "|" + historic_df["schedule_week"].astype(str)
         + "|" + historic_df["team_home"].astype(str)
         + "|" + historic_df["team_away"].astype(str))
        .values
    )

    new_rows_df = new_rows_df[~new_rows_df["__key"].isin(hist_keys)].drop(columns=["__key"])  # anti-join
    if new_rows_df.empty:
        return 0

    # Assign sequential index values continuing from existing max index
    try:
        start_index = int(pd.to_numeric(pd.Series(historic_df.index)).max())
    except Exception:
        # If index isn't numeric for some reason, fall back to length-1
        start_index = len(historic_df) - 1

    new_index = list(range(start_index + 1, start_index + 1 + len(new_rows_df)))
    new_rows_df.index = new_index
    new_rows_df.index.name = ""  # keep blank index header

    # Concatenate and persist
    out_df = pd.concat([historic_df, new_rows_df], axis=0)
    out_df.index.name = ""
    out_df.to_csv(historic_csv)

    return len(new_rows_df)

# Example usage (uncomment to run):
# appended = append_week_lines_to_historic()
# print(f"Appended {appended} rows to historic_lines.csv")



In [429]:
appended = append_week_lines_to_historic()

In [437]:
week_3 = pd.read_csv('data/predictions_week_3.csv')
week_3 = week_3[['player_id', 'player_display_name', 'position', 'team', 'opponent_team', 'predicted_touchdown_probability', 'price', 'model_edge', 'market_implied_prob']]
top_rb = week_3[week_3['position'] == 'RB'].sort_values('predicted_touchdown_probability', ascending=False).head(10)
top_wr = week_3[week_3['position'] == 'WR'].sort_values('predicted_touchdown_probability', ascending=False).head(10)
top_te = week_3[week_3['position'] == 'TE'].sort_values('predicted_touchdown_probability', ascending=False).head(5)
top_qb = week_3[week_3['position'] == 'QB'].sort_values('predicted_touchdown_probability', ascending=False).head(5)
ev = week_3[week_3['model_edge'] >= 0.045]
ev = ev[ev['price'] <= 400]
ev = ev.sort_values('model_edge', ascending=False)
high_prob = week_3[week_3['predicted_touchdown_probability'] >= 0.50]








In [438]:
top_rb

,player_id,player_display_name,position,team,opponent_team,predicted_touchdown_probability,price,model_edge,market_implied_prob
0,00-0032764,Derrick Henry,RB,BAL,DET,0.591463,-215.0,-0.091076,0.682540
1,00-0039139,Jahmyr Gibbs,RB,DET,BAL,0.563612,-120.0,0.018157,0.545455
2,00-0037248,James Cook,RB,BUF,MIA,0.559112,-175.0,-0.077251,0.636364
3,00-0038542,Bijan Robinson,RB,ATL,CAR,0.545368,-185.0,-0.103755,0.649123
4,00-0034844,Saquon Barkley,RB,PHI,LA,0.539435,-160.0,-0.075950,0.615385
5,00-0035700,Josh Jacobs,RB,GB,CLE,0.531103,-140.0,-0.052230,0.583333
6,00-0036223,Jonathan Taylor,RB,IND,TEN,0.511606,-195.0,-0.149411,0.661017
7,00-0036555,Chuba Hubbard,RB,CAR,ATL,0.484769,140.0,0.068103,0.416667
8,00-0035685,David Montgomery,RB,DET,BAL,0.481927,120.0,0.027382,0.454545
9,00-0037840,Kyren Williams,RB,LA,PHI,0.472042,-105.0,-0.040153,0.512195


In [439]:
top_wr

,player_id,player_display_name,position,team,opponent_team,predicted_touchdown_probability,price,model_edge,market_implied_prob
14,00-0036358,CeeDee Lamb,WR,DAL,CHI,0.437029,-110.0,-0.086781,0.523810
16,00-0036963,Amon-Ra St. Brown,WR,DET,BAL,0.428684,135.0,0.003152,0.425532
17,00-0038544,Quentin Johnston,WR,LAC,DEN,0.421260,230.0,0.118230,0.303030
18,00-0030279,Keenan Allen,WR,LAC,DEN,0.403015,195.0,0.064032,0.338983
19,00-0039064,Zay Flowers,WR,BAL,DET,0.398218,115.0,-0.066898,0.465116
20,00-0038543,Jaxon Smith-Njigba,WR,SEA,NO,0.397282,110.0,-0.078909,0.476190
21,00-0036912,DeVonta Smith,WR,PHI,LA,0.395531,210.0,0.072950,0.322581
22,00-0039919,Rome Odunze,WR,CHI,DAL,0.395522,195.0,0.056539,0.338983
23,00-0039075,Puka Nacua,WR,LA,PHI,0.393454,145.0,-0.014709,0.408163
25,00-0039915,Ladd McConkey,WR,LAC,DEN,0.385191,115.0,-0.079925,0.465116


In [440]:
top_te






,player_id,player_display_name,position,team,opponent_team,predicted_touchdown_probability,price,model_edge,market_implied_prob
32,00-0037744,Trey McBride,TE,ARI,SF,0.364803,165.0,-0.012556,0.377358
34,00-0040128,Tyler Warren,TE,IND,TEN,0.361256,220.0,0.048756,0.312500
37,00-0038996,Tucker Kraft,TE,GB,CLE,0.358079,200.0,0.024745,0.333333
38,00-0039338,Brock Bowers,TE,LV,WAS,0.354011,110.0,-0.122180,0.476190
47,00-0030506,Travis Kelce,TE,KC,NYG,0.336617,140.0,-0.080050,0.416667


In [441]:
top_qb

,player_id,player_display_name,position,team,opponent_team,predicted_touchdown_probability,price,model_edge,market_implied_prob
30,00-0036389,Jalen Hurts,QB,PHI,LA,0.374844,-135.0,-0.199624,0.574468
64,00-0034857,Josh Allen,QB,BUF,MIA,0.292528,-160.0,-0.322856,0.615385
65,00-0034796,Lamar Jackson,QB,BAL,DET,0.288944,130.0,-0.145839,0.434783
69,00-0033873,Patrick Mahomes,QB,KC,NYG,0.280939,265.0,0.006966,0.273973
83,00-0039851,Drake Maye,QB,NE,PIT,0.229022,280.0,-0.034136,0.263158


In [442]:
ev

,player_id,player_display_name,position,team,opponent_team,predicted_touchdown_probability,price,model_edge,market_implied_prob
17,00-0038544,Quentin Johnston,WR,LAC,DEN,0.421260,230.0,0.118230,0.303030
67,00-0033885,David Njoku,TE,CLE,GB,0.284732,380.0,0.076399,0.208333
29,00-0038117,Wan'Dale Robinson,WR,NYG,KC,0.376525,230.0,0.073494,0.303030
21,00-0036912,DeVonta Smith,WR,PHI,LA,0.395531,210.0,0.072950,0.322581
7,00-0036555,Chuba Hubbard,RB,CAR,ATL,0.484769,140.0,0.068103,0.416667
18,00-0030279,Keenan Allen,WR,LAC,DEN,0.403015,195.0,0.064032,0.338983
22,00-0039919,Rome Odunze,WR,CHI,DAL,0.395522,195.0,0.056539,0.338983
34,00-0040128,Tyler Warren,TE,IND,TEN,0.361256,220.0,0.048756,0.312500


In [443]:
high_prob

,player_id,player_display_name,position,team,opponent_team,predicted_touchdown_probability,price,model_edge,market_implied_prob
0,00-0032764,Derrick Henry,RB,BAL,DET,0.591463,-215.0,-0.091076,0.682540
1,00-0039139,Jahmyr Gibbs,RB,DET,BAL,0.563612,-120.0,0.018157,0.545455
2,00-0037248,James Cook,RB,BUF,MIA,0.559112,-175.0,-0.077251,0.636364
3,00-0038542,Bijan Robinson,RB,ATL,CAR,0.545368,-185.0,-0.103755,0.649123
4,00-0034844,Saquon Barkley,RB,PHI,LA,0.539435,-160.0,-0.075950,0.615385
5,00-0035700,Josh Jacobs,RB,GB,CLE,0.531103,-140.0,-0.052230,0.583333
6,00-0036223,Jonathan Taylor,RB,IND,TEN,0.511606,-195.0,-0.149411,0.661017


In [489]:
weekly_data = nfl.import_weekly_data(years=[2017])
#find players that scored in weeks 1-3
weekly_data = weekly_data[weekly_data['week'].isin([1,2,3])]

weekly_data['tds'] = weekly_data['receiving_tds'] + weekly_data['rushing_tds']

scorers = weekly_data[weekly_data['tds'] > 0]

scorers[scorers['player_display_name'].isin(
    scorers['player_display_name'].value_counts()[lambda x: x >= 3].index
)].player_display_name.unique()






Downcasting floats.


array(['Chris Thompson', 'Devonta Freeman', 'Melvin Gordon',
       'Todd Gurley', 'Leonard Fournette', 'Kareem Hunt'], dtype=object)

In [490]:
weekly_data = pd.read_csv('data/stats_player_week_2025.csv')
#find players that scored in weeks 1-3
weekly_data = weekly_data[weekly_data['week'].isin([1,2,3])]

weekly_data['tds'] = weekly_data['receiving_tds'] + weekly_data['rushing_tds']

scorers = weekly_data[weekly_data['tds'] > 0]

scorers[scorers['player_display_name'].isin(
    scorers['player_display_name'].value_counts()[lambda x: x >= 2].index
)].player_display_name.unique()


array(['Zach Ertz', 'Keenan Allen', 'DeAndre Hopkins', 'James Conner',
       'Patrick Mahomes', 'Saquon Barkley', 'Josh Jacobs', 'Daniel Jones',
       'Deebo Samuel Sr.', 'J.K. Dobbins', 'Jalen Hurts', 'Chuba Hubbard',
       'Javonte Williams', 'James Cook', 'Quentin Johnston',
       'Cedric Tillman', 'Tucker Kraft', "De'Von Achane", 'Davis Allen',
       'Rome Odunze', 'Emeka Egbuka'], dtype=object)